In [2]:
# Imports
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import datetime

In [3]:
# Read in the geojson file for Chicago neighborhoods
geo_file = "./chicago.geojson"
neighborhoods_gdf = gpd.read_file(geo_file)

In [4]:
# Read the original file from dataset into pandas
og_file = "./data/traffic_crashes.csv"
og_df = pd.read_csv(og_file)
og_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 746498 entries, 0 to 746497
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH_RECORD_ID                746498 non-null  object 
 1   RD_NO                          742191 non-null  object 
 2   CRASH_DATE_EST_I               56389 non-null   object 
 3   CRASH_DATE                     746498 non-null  object 
 4   POSTED_SPEED_LIMIT             746498 non-null  int64  
 5   TRAFFIC_CONTROL_DEVICE         746498 non-null  object 
 6   DEVICE_CONDITION               746498 non-null  object 
 7   WEATHER_CONDITION              746498 non-null  object 
 8   LIGHTING_CONDITION             746498 non-null  object 
 9   FIRST_CRASH_TYPE               746498 non-null  object 
 10  TRAFFICWAY_TYPE                746498 non-null  object 
 11  LANE_CNT                       199004 non-null  float64
 12  ALIGNMENT                     

In [5]:
# Reduce dataset
new_df = og_df[['RD_NO', 'CRASH_DATE', 'POSTED_SPEED_LIMIT', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 
                                    'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'CRASH_TYPE', 'DAMAGE', 'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NAME', 
                                    'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'LATITUDE', 'LONGITUDE', 'LOCATION']]
new_df = new_df.dropna()

In [6]:
# Change date to datetime
new_df['CRASH_DATE'] = pd.to_datetime(new_df['CRASH_DATE'])


/var/folders/b_/47k771614dlgjd4cqk_f_y180000gn/T/ipykernel_31367/2507884040.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  new_df['CRASH_DATE'] = pd.to_datetime(new_df['CRASH_DATE'])


In [7]:
# Select only the desired years
selected_years = [2018, 2019, 2020, 2021, 2022]
years_df = new_df[new_df['CRASH_DATE'].dt.year.isin(selected_years)]
years_df = years_df.sort_values(by='CRASH_DATE', ascending=True)
years_df.head()


,RD_NO,CRASH_DATE,POSTED_SPEED_LIMIT,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,ROADWAY_SURFACE_COND,ROAD_DEFECT,CRASH_TYPE,DAMAGE,PRIM_CONTRIBUTORY_CAUSE,SEC_CONTRIBUTORY_CAUSE,STREET_NAME,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
718580,JB100559,2018-01-01 00:00:00,20,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,NOT DIVIDED,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,"OVER $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,HARPER AVE,0,2,1,41.800575,-87.589225,POINT (-87.589225075066 41.80057461981)
13920,JB100868,2018-01-01 00:00:00,25,UNKNOWN,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,"$501 - $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,117TH ST,0,2,1,41.681735,-87.641014,POINT (-87.641013916869 41.681735373094)
175227,JB100763,2018-01-01 00:00:00,30,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,ONE-WAY,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,"$501 - $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,CUYLER AVE,0,2,1,41.955842,-87.650268,POINT (-87.65026813462 41.955842185147)
637953,JB100036,2018-01-01 00:05:00,30,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,"$501 - $1,500",UNABLE TO DETERMINE,UNABLE TO DETERMINE,ASHLAND AVE,0,2,1,41.949417,-87.668774,POINT (-87.66877431033 41.949417412921)
381931,JB100179,2018-01-01 00:07:00,35,SNOW,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,NOT DIVIDED,SNOW OR SLUSH,NO DEFECTS,NO INJURY / DRIVE AWAY,"OVER $1,500",WEATHER,FAILING TO REDUCE SPEED TO AVOID CRASH,STATE ST,0,2,1,41.683946,-87.622993,POINT (-87.622993129041 41.683945885086)


# GeoPandas

In [8]:
# Set coordinate points from DataFrame to variables, create geoDataFrame
geometry = [Point(lon, lat) for lon, lat in zip(years_df['LONGITUDE'],
                                                years_df['LATITUDE'])]
geo_points_gdf = gpd.GeoDataFrame(years_df, geometry=geometry)

In [9]:
# Apply the Coordinate Reference System (CRS) from neigborhoods_gdf
geo_points_gdf.crs = neighborhoods_gdf.crs

In [10]:
# Combine the geoDataFrames to place accident locations inside neighborhoods
combined_gdf = gpd.sjoin(geo_points_gdf, neighborhoods_gdf, how='left', op='within')
combined_gdf = combined_gdf.reset_index(drop=True)

/opt/anaconda3/envs/dev/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3517: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [11]:
# Select the needed columns
reduced_gdf = combined_gdf[['CRASH_DATE',
                            'FIRST_CRASH_TYPE',
                            'geometry',
                            'name']]
                                        

In [12]:
# Seperate the data by year
gdf_2018 = reduced_gdf[reduced_gdf['CRASH_DATE'].dt.year == 2018].reset_index(drop=True)
gdf_2019 = reduced_gdf[reduced_gdf['CRASH_DATE'].dt.year == 2019].reset_index(drop=True)
gdf_2020 = reduced_gdf[reduced_gdf['CRASH_DATE'].dt.year == 2020].reset_index(drop=True)
gdf_2021 = reduced_gdf[reduced_gdf['CRASH_DATE'].dt.year == 2021].reset_index(drop=True)
gdf_2022 = reduced_gdf[reduced_gdf['CRASH_DATE'].dt.year == 2022].reset_index(drop=True)


In [13]:
# 2018 group by neigborhood name and count
crash_count_2018 = gdf_2018.groupby(['name'])\
.size().reset_index(name='count')
sorted_2018_count = crash_count_2018\
.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)


In [14]:
# 2019 group by neighborhood name and count
crash_count_2019 = gdf_2019.groupby(['name'])\
.size().reset_index(name='count')
sorted_2019_count = crash_count_2019\
.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)


In [15]:
# 2020 group by neighborhood name and count
crash_count_2020 = gdf_2020.groupby(['name'])\
.size().reset_index(name='count')
sorted_2020_count = crash_count_2020\
.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)


In [16]:
# 2021 group by neighborhood name and count
crash_count_2021 = gdf_2021.groupby(['name'])\
.size().reset_index(name='count')
sorted_2021_count = crash_count_2021\
.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)


In [17]:
# 2022 group by neighborhood name and count
crash_count_2022 = gdf_2022.groupby(['name'])\
.size().reset_index(name='count')
sorted_2022_count = crash_count_2020\
.sort_values(by=['name', 'count'], ascending=[True, False]).reset_index(drop=True)


### Write to CSV

In [18]:
sorted_2018_count.to_csv('./geo_yearly_data/sorted_2018_count.csv', index=False, header=True)
sorted_2019_count.to_csv('./geo_yearly_data/sorted_2019_count.csv', index=False, header=True)
sorted_2020_count.to_csv('./geo_yearly_data/sorted_2020_count.csv', index=False, header=True)
sorted_2021_count.to_csv('./geo_yearly_data/sorted_2021_count.csv', index=False, header=True)
sorted_2022_count.to_csv('./geo_yearly_data/sorted_2022_count.csv', index=False, header=True)